# Kateto Finetune — RWKV7-G1 2.9B LoRA en Colab (GPU)

Corrés todo a mano, de arriba hacia abajo. Espejo de `run_kateto.sh` pero en GPU:
`--accelerator gpu --devices 1 --precision bf16 --op cuda`.
Base: `rwkv7-g1j-2.9b-20260831-ctx16384.pth` de `BlinkDL/rwkv7-g1` (~5.5GB).
Datos: `sft_train.jsonl` (
... [cell truncated in skill-update payload]

In [ ]:
# 1) Instalar dependencias (espejo de RWKV-PEFT/requirements.txt, con pins que funcionan juntos)
# NOTA: los avisos de gradio/diffusers vs huggingface-hub son inofensivos (paquetes de Colab, el train no los usa).
# La GPU se verifica por torch abajo, no por nvidia-smi (no siempre esta en el PATH).
!pip install -q transformers==4.57.1 peft==0.17.1 lightning==2.6.5
!pip install -q deepspeed einops triton datasets jsonlines huggingface_hub bitsandbytes
!pip install -q rwkv-fla rwkv
!python -c "import torch; print('torch', torch.__version__, '| cuda_disponible:', torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'SIN GPU: Entorno > Cambiar tipo de entorno > T4 GPU, reinicia el entorno y reintenta')"
!python -c "import transformers, peft, lightning; print(transformers.__version__, peft.__version__, lightning.__version__)"


In [ ]:
# 2) Descargar el pth base desde HuggingFace (~5.5GB)
from huggingface_hub import hf_hub_download
import os
os.makedirs('/content/base', exist_ok=True)
p = hf_hub_download(repo_id='BlinkDL/rwkv7-g1', filename='rwkv7-g1j-2.9b-20260831-ctx16384.pth', local_dir='/content/base')
print(p, os.path.getsize(p))

In [ ]:
# 3) Subir sft_train.jsonl desde tu PC
from google.colab import files
import os, json
print('Subí sft_train.jsonl (618 filas, campos query/response):')
up = files.upload()
# Normaliza el nombre por si el browser le cambia algo
src = 'sft_train.jsonl' if 'sft_train.jsonl' in up else list(up.keys())[0]
os.replace(src, '/content/sft_train.jsonl')
n = sum(1 for _ in open('/content/sft_train.jsonl'))
r0 = json.loads(open('/content/sft_train.jsonl').readline())
print(f'filas={n} keys={sorted(r0.keys())}')
assert n == 618, f'esperaba 618 filas, veo {n}'
assert set(['query', 'response']) <= set(r0), 'faltan campos query/response'

In [ ]:
# 4) Obtener el codigo RWKV-PEFT (fork de Joluck, el mismo que esta probado localmente)
!test -d /content/RWKV-PEFT || git clone https://github.com/Joluck/RWKV-PEFT /content/RWKV-PEFT || (cd /content && rm -rf RWKV-PEFT-main main.zip && wget -q https://github.com/Joluck/RWKV-PEFT/archive/refs/heads/main.zip -O main.zip && unzip -q main.zip && mv RWKV-PEFT-main RWKV-PEFT)
!ls /content/RWKV-PEFT/train.py


In [ ]:
# 5) Smoke test: epoch_steps 5 (misma config que run_kateto.sh, en GPU)
!cd /content/RWKV-PEFT && python train.py \
  --load_model /content/base/rwkv7-g1j-2.9b-20260831-ctx16384.pth \
  --proj_dir /content/out-smoke --data_file /content/sft_train.jsonl \
  --vocab_size 65536 --n_layer 32 --n_embd 2560 \
  --ctx_len 1024 --micro_bsz 1 --accumulate_grad_batches 16 \
  --epoch_steps 5 --epoch_count 1 --epoch_begin 0 --epoch_save 1 \
  --lr_init 1e-4 --lr_final 1e-5 --warmup_steps 50 --beta1 0.9 --beta2 0.99 \
  --grad_cp 1 --dropout 0.0 --weight_decay 0.01 \
  --accelerator gpu --devices 1 --precision bf16 \
  --my_testing x070 --op cuda \
  --peft lora --lora_config '{"lora_load":"", "lora_r":16, "lora_alpha":32, "lora_dropout":0.05}' \
  --data_type sft --sft_field query response --sft_split train

In [ ]:
# 6) Corrida full: epoch_steps 500, lr 1e-4 -> 1e-5, warmup 50
!cd /content/RWKV-PEFT && python train.py \
  --load_model /content/base/rwkv7-g1j-2.9b-20260831-ctx16384.pth \
  --proj_dir /content/out --data_file /content/sft_train.jsonl \
  --vocab_size 65536 --n_layer 32 --n_embd 2560 \
  --ctx_len 1024 --micro_bsz 1 --accumulate_grad_batches 16 \
  --epoch_steps 500 --epoch_count 1 --epoch_begin 0 --epoch_save 1 \
  --lr_init 1e-4 --lr_final 1e-5 --warmup_steps 50 --beta1 0.9 --beta2 0.99 \
  --grad_cp 1 --dropout 0.0 --weight_decay 0.01 \
  --accelerator gpu --devices 1 --precision bf16 \
  --my_testing x070 --op cuda \
  --peft lora --lora_config '{"lora_load":"", "lora_r":16, "lora_alpha":32, "lora_dropout":0.05}' \
  --data_type sft --sft_field query response --sft_split train

In [ ]:
# 7) Guardar LoRA y descargarlo
!ls -lh /content/out/*.pth | tail -5
!cd /content && zip -j kateto-lora.zip out/*.pth && ls -lh kateto-lora.zip
from google.colab import files
files.download('/content/kateto-lora.zip')